# 77 — Campaign C S0: rank / gap / partial-budget sweep

設計書 `campaign_c_m5_obligation_closure_detailed_design.md` の **Part I / S0**。

- 現行 `-v2` M3/M4/M5 は **変更しない**（archive）
- 同一 M2 parent 上で **nested high-rank RSVD** を1回だけ実行し、rank を切って評価
- すべて `HEURISTIC_EXPLORATORY_NOT_A_RIGOROUS_BOUND`
- 結果を M5 ledger に直接入れない
- 選択 rank があるときだけ次の notebook 78 へ


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'rank_sweep.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/rank_sweep.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
os.environ.setdefault('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('S0 nested RSVD for j2_max=2 expects CUDA on Paperspace.')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.common import read_json

M7C_RUN_ID = os.environ['VALIDATED_RG_M7C_RUN_ID']
CAND = os.environ['VALIDATED_RG_STAGED_CANDIDATE']
PACKAGE_ROOT = Path(
    os.environ.get('VALIDATED_RG_STAGED_PACKAGE')
    or (PERSIST_ROOT / 'searches' / M7C_RUN_ID / 'auto_execute' / CAND)
).expanduser().resolve()
CHILD_IDS = read_json(PACKAGE_ROOT / 'child_run_ids.json')
M2_RUN_ID = CHILD_IDS['M2']
print('PACKAGE_ROOT', PACKAGE_ROOT)
print('M2_RUN_ID', M2_RUN_ID)
print('archived_M3', CHILD_IDS.get('M3'))
print('archived_M4', CHILD_IDS.get('M4'))
print('archived_M5', CHILD_IDS.get('M5'))


## M3Config（child M2 親）+ sweep config

A4000 16GB 向け既定: `max_factor_rank≈96`、grid はそれ以下。必要なら env で上書き。


In [ ]:
from dataclasses import asdict
from pathlib import Path
from src.common import read_json
from src.cutoff_dims import cutoff_dimension_payload
from src.m3_config import M3Config
from src.m7_staged_lineage import write_child_m2_acceptance_audit
from src.rank_sweep import default_sweep_config

over = read_json(PACKAGE_ROOT / 'm3_config_overrides.json')
audit_path = PROJECT_ROOT / 'audit' / 'm2_accepted_parent.json'
audit = read_json(audit_path) if audit_path.is_file() else None
if not isinstance(audit, dict) or audit.get('accepted_run_id') != M2_RUN_ID:
    audit = write_child_m2_acceptance_audit(
        PROJECT_ROOT, run_root=PERSIST_ROOT / 'runs' / M2_RUN_ID,
    )
dims = cutoff_dimension_payload(int(over['j2_max']))
base = asdict(M3Config())
base.update({
    'parent_run_id': audit['accepted_run_id'],
    'parent_checkpoint': Path(audit['checkpoint_path']).name,
    'parent_checkpoint_path': audit['checkpoint_path'],
    'parent_report_path': audit['m2_report_path'],
    'parent_acceptance_path': audit['m2_acceptance_path'],
    'j2_max': int(over['j2_max']),
    'sector_count': int(over['sector_count']),
    'operator_dimension': int(over['operator_dimension']),
    'target_rank': int(over['target_rank']),
    'require_cuda': True,
})
M3_CONFIG = M3Config(**base)

# Practical Paperspace defaults; override via env JSON if needed.
SWEEP_CONFIG = default_sweep_config(
    max_factor_rank=int(os.environ.get('VALIDATED_RG_SWEEP_MAX_RANK', '96')),
    rank_grid=[16, 24, 25, 32, 36, 48, 49, 64, 81, 96],
    oversampling=16,
    power_iterations=2,
    seed=20260720,
    engineering_margin=0.05,
    sectors_per_shard=8,
)
print(json.dumps({
    'parent_m2': M3_CONFIG.parent_run_id,
    'j2_max': M3_CONFIG.j2_max,
    'operator_dimension': M3_CONFIG.operator_dimension,
    'dims': dims,
    'sweep': {k: SWEEP_CONFIG[k] for k in (
        'max_factor_rank', 'rank_grid', 'engineering_margin', 'seed',
    )},
}, indent=2, ensure_ascii=False))


## 実行 S0 sweep（checkpoint 不要・探索専用）


In [ ]:
from src.rank_sweep import run_rank_sweep
from src.rank_sweep_reporting import write_rank_sweep_report
from src.runtime import validate_persistent_root

validate_persistent_root()
SUMMARY = run_rank_sweep(
    project_root=PROJECT_ROOT,
    persistent_root=PERSIST_ROOT,
    package_root=PACKAGE_ROOT,
    m3_config=M3_CONFIG,
    candidate_id=CAND,
    sweep_config=SWEEP_CONFIG,
)
PATHS = write_rank_sweep_report(Path(SUMMARY['sweep_root']))
print('sweep_root', SUMMARY['sweep_root'])
print('selection_status', SUMMARY['selection_status'])
print('selected_rank', SUMMARY['selected_rank'])
print(json.dumps({
    'selection_status': SUMMARY['selection_status'],
    'selected_rank': SUMMARY['selected_rank'],
    'feasible_ranks': SUMMARY.get('feasible_ranks'),
    'selection_reasons': SUMMARY.get('selection_reasons'),
    'paths': PATHS,
    'status': SUMMARY['status'],
}, indent=2, ensure_ascii=False))
assert SUMMARY['status'] == 'EXPLORATORY_NOT_CERTIFIED'


## 判定

- `SELECTED` → notebook **78** で proof configuration freeze + 新 M3-R
- `NO_SELECTION` / `REJECT_CANDIDATE` → candidate archive（M6 禁止）


In [ ]:
from src.m7_archive import archive_from_sweep, write_advance
from pathlib import Path as _Path

selection = SUMMARY['selection_status']
if selection == 'SELECTED':
    write_advance(
        PACKAGE_ROOT,
        selected_rank=int(SUMMARY['selected_rank']),
        sweep_root=SUMMARY['sweep_root'],
        selection_reasons=list(SUMMARY.get('selection_reasons') or []),
    )
    print('NEXT: open notebooks/78_m3_rigorous_rank_candidate.ipynb')
    print('freeze selected_rank', SUMMARY['selected_rank'])
elif selection in {'NO_SELECTION', 'REJECT_CANDIDATE'}:
    archive_from_sweep(
        PACKAGE_ROOT,
        _Path(SUMMARY['sweep_root']),
        selection_status=selection,
        selection_reasons=list(SUMMARY.get('selection_reasons') or []),
    )
    print('ARCHIVED candidate; do not start M6. Inspect rank_sweep_summary.md')
    print('NEXT: notebooks/82_campaign_c_candidate_queue.ipynb')
else:
    print('Unexpected selection_status', selection)
